In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import numpy as np
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test-other/LibriSpeech/test-other/"
DESTINATION = "/workspaces/dev/output/LibriSpeechASRcorpus/sclient/rt_whisper/test-other/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
HYPERPARAMETER_PATH = "./hyperparameters/sclite.yml"

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(hyperparameter_path=HYPERPARAMETER_PATH)

In [ ]:
src = Path(SOURCE)
dest = Path(DESTINATION)
dest.mkdir(parents=True, exist_ok=True)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    completed = []
    param = Param()
    # start_time = time.perf_counter()
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        result:Result = token_streamer.process(param)
        completed.extend(result.completed)
        param.update(result)
    completed.extend(result.candidate)
    # end_time = time.perf_counter()
    # print(f"Processed {flac.stem} in {end_time - start_time:.2f} seconds")

    return TRNFormat(
        id = flac.stem,
        text = normalize_text_only_en(
            " ".join([s.text for s in completed])
        ).upper()
    )

In [ ]:
audio_list = list(src.rglob("*.flac"))

1688-142285-0000  
    THERE'S IRON THEY SAY IN ALL OUR BLOOD AND A GRAIN OR TWO PERHAPS IS GOOD BUT HIS HE MAKES ME HARSHLY FEEL HAS GOT A LITTLE TOO MUCH OF STEEL ANON  
1688-142285-0001  
    MARGARET SAID MISTER HALE AS HE RETURNED FROM SHOWING HIS GUEST DOWNSTAIRS I COULD NOT HELP WATCHING YOUR FACE WITH SOME ANXIETY WHEN MISTER THORNTON MADE HIS CONFESSION OF HAVING BEEN A SHOP BOY  
1688-142285-0002  
    YOU DON'T MEAN THAT YOU THOUGHT ME SO SILLY  
1688-142285-0003  
    I REALLY LIKED THAT ACCOUNT OF HIMSELF BETTER THAN ANYTHING ELSE HE SAID  
1688-142285-0004  
    HIS STATEMENT OF HAVING BEEN A SHOP BOY WAS THE THING I LIKED BEST OF ALL  
1688-142285-0005   
    YOU WHO WERE ALWAYS ACCUSING PEOPLE OF BEING SHOPPY AT HELSTONE  

In [ ]:
flac = audio_list[1]
audio = librosa.load(flac, sr=SAMPLE_RATE)[0]
trn_format = transcriber(flac)

In [ ]:
trn_format.text